# 05 — Inference and prediction saving

**Purpose:** Load a saved LoRA adapter or a fully tuned model, generate one category per example, and save raw outputs and metadata.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


Run once per trained checkpoint and language. **Only change `SPLIT` to `test` after finalizing training settings.** Run the English test on the same checkpoint if examining retention. The baseline model may use this notebook with `METHOD="base"`.

In [ ]:
import torch
TASK='sib200'; LANG='ur'; SPLIT='validation'  # finally: 'test'
METHOD='lora'  # 'lora', 'full', or 'base'
RUN_ID='sib200_lora_ur_n10_s42_r8'
MAX_ROWS=None  # 10 for smoke, None for full split
labels = task_labels(TASK)
rows = load_task(TASK,LANG,SPLIT)
if MAX_ROWS is not None: rows=rows[:MAX_ROWS]
model_id=cfg()['model_id']; checkpoint=None
if METHOD!='base':
    state=read_json(ROOT/'outputs/checkpoints'/RUN_ID/'complete.json')
    if state['method'] != METHOD or state['task'] != TASK: raise ValueError('Checkpoint metadata does not match method/task')
    model_id=state['base_model']; checkpoint=Path(state['saved_path'])
    if not checkpoint.exists(): checkpoint=ROOT/'outputs/checkpoints'/RUN_ID/'model'
    if not checkpoint.exists(): raise FileNotFoundError(checkpoint)
    if METHOD=='full':
        model,tokenizer=load_model(str(checkpoint))
    else:
        model,tokenizer=load_model(model_id,adapter_path=str(checkpoint))
else:
    model,tokenizer=load_model(model_id)
print('Model:',model_id,'mode:',METHOD,'rows:',len(rows))

In [ ]:
path=ROOT/'outputs/predictions'/f'{RUN_ID}_{LANG}_{SPLIT}.csv'
meta=dict(task=TASK,lang=LANG,split=SPLIT,model_id=model_id,
          run_id=RUN_ID,method=METHOD,checkpoint=str(checkpoint) if checkpoint else None,
          dataset_sha256=dataset_sha(TASK,LANG,SPLIT),subset=(MAX_ROWS is not None),
          decoding={'do_sample':False,'max_new_tokens':cfg()['max_new_tokens']})
outputs=infer_resume(model,tokenizer,rows,labels,LANG,path,meta,cfg()['max_new_tokens'])
print('Saved:',path, 'count:',len(outputs),'invalid:',sum(not x['valid'] for x in outputs))
print('First outputs:',[(p['reference'],p['prediction'],p['raw_prediction']) for p in outputs[:8]])

In [ ]:
del model
if torch.cuda.is_available(): torch.cuda.empty_cache()